# **Datasets**

In this notebook the exploratory data analysis is done for the datasets used in the [TAD-GAN](https://arxiv.org/pdf/2009.07769) study. 

The datasets included are **spacecraft telemetry** signals from NASA, **Yahoo S5** and **Numenta Anomaly Benchmark**.

In [ ]:
import os
import pandas as pd
import numpy as np
import json
import kagglehub
import matplotlib.pyplot as plt

In [ ]:
def plot_anomaly_df(
    df: pd.DataFrame, 
    save: bool = False, 
    save_jpg: str = ""
):
    df['is_anomaly'] = df['is_anomaly'].astype(bool)

    plt.figure(figsize=(5, 2))

    plt.plot(
        df['value'].values,  # .values strips any custom index issues
        color='black',
        linewidth=1.5
    )

    anomaly_regions = []
    start = None

    for i in range(len(df)):
        if df['is_anomaly'].iloc[i]:
            if start is None:
                start = i
        else:
            if start is not None:
                end = i - 1
                anomaly_regions.append((start, end))
                start = None

    if start is not None:
        anomaly_regions.append((start, len(df) - 1))

    for start, end in anomaly_regions:
        plt.axvspan(start, end, color='red', alpha=0.25)

    plt.xlabel("")
    plt.ylabel("")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    if save:
        save_jpg = (
            f"anomaly_plot_{np.random.randint(low=10_000_000, high=99_999_999)}.png"
            if save_jpg == ""
            else save_jpg
        )
        plt.savefig(save_jpg, dpi=300, bbox_inches='tight')

    plt.show()

## **NASA**

The NASA data contains *spacecraft telemetry* signals, consisting of two datasets:
- *Mars Science Laboratory* (MSL)
- *Soil Moisture Active Passive* (SMAP)

The split of the dataset was taken from the authors of the paper [Multivariate Time-series Anomaly Detection via
Graph Attention Network](https://arxiv.org/pdf/2009.02040), more precisely from their [github](https://github.com/ML4ITS/mtad-gat-pytorch/tree/main) implementation. 

In the original paper that published these two datasets, the anomalies were categorized as:
- *point*: that can be identified by properly-set thresholds and distance based methods, and don't require temporal data
- *contextual*: those that might require more *complex* methodologies (i.e., LSTM, HTM) approaches to detect them

In TadGAN, these two types of anomalies are merged and used as contextual anomalies.

In [ ]:
NASA_SPLIT_REPO = "https://raw.githubusercontent.com/ML4ITS/mtad-gat-pytorch/refs/heads/main/datasets/data/"
NASA_LABELS_FILE = "labeled_anomalies.csv"
NASA_MSL_SPLIT = "msl_train_md.csv"
NASA_SMAP_SPLIT = "smap_train_md.csv"

In [ ]:
nasa_dataset_path = kagglehub.dataset_download("patrickfleith/nasa-anomaly-detection-dataset-smap-msl")
nasa_train_path = os.path.join(nasa_dataset_path, "data", "data", "train")
nasa_test_path = os.path.join(nasa_dataset_path, "data", "data", "test")

In [ ]:
nasa_dataset_dir = os.path.join(os.getcwd(), "datasets", "nasa")

if os.path.exists(nasa_dataset_dir):
    print("Folder for the NASA datasets already exists!")
else:
    os.mkdir(nasa_dataset_dir)
    print("Folder for the NASA datasets was succesfully created!")

In [ ]:
nasa_labels_path = os.path.join(nasa_dataset_dir, "labels.csv")
nasa_labels_url = f"{NASA_SPLIT_REPO}{NASA_LABELS_FILE}"
nasa_labels = pd.read_csv(nasa_labels_url)
nasa_labels.to_csv(nasa_labels_path)

### **MSL Dataset**

In the TAD-GAN research, the train and test datasets were concatenated.

In [ ]:
msl_split_path = os.path.join(nasa_dataset_dir, "msl.csv")
msl_split_url = f"{NASA_SPLIT_REPO}{NASA_MSL_SPLIT}"
msl_split = pd.read_csv(msl_split_url)
msl_split.to_csv(msl_split_path)

# extract the file names corresponding to the MSL dataset
msl_file_names = list(msl_split["chan_id"].values) 
# add the .npy file extensions to the MSL file names
msl_file_names = [f"{msl_file_name}.npy" for msl_file_name in msl_file_names]
# sort them alphabetically
msl_file_names.sort()
print(msl_file_names)

In [ ]:
msl = []

for msl_file_name in msl_file_names:
    # load both the train and test time series for the same channel, concatenate
    # the time series and add them to the MSL dataset
    msl_train_ts = np.load(os.path.join(nasa_train_path, msl_file_name))
    msl_test_ts = np.load(os.path.join(nasa_test_path, msl_file_name))
    msl_ts = np.concatenate((msl_train_ts, msl_test_ts), axis=0)
    
    msl.append(msl_ts)

print(f"MSL Total Telemetry Channels: {len(msl)}")

In [ ]:
msl_labels = [None for _ in range(len(msl))]
msl_labels_data = nasa_labels[nasa_labels["spacecraft"] == "MSL"]
msl_total_anomaly_windows = 0

for index, row in msl_labels_data.iterrows():
    # extract the telemetry from which the anomalies were obtained,
    # and search for its corresponding index in the MSL dataset
    telemetry = f"{row['chan_id']}.npy"
    telemetry_idx = msl_file_names.index(telemetry)
    if msl_labels[telemetry_idx] is None: 
        msl_labels[telemetry_idx] = np.zeros((msl[telemetry_idx].shape[0],))

    anomaly_sequences = json.loads(row["anomaly_sequences"])
    msl_total_anomaly_windows += len(anomaly_sequences)

    for start, end in anomaly_sequences:
        msl_labels[telemetry_idx][start:end + 1] = 1

msl_labels = np.hstack(msl_labels)
msl = np.concatenate(msl, axis=0)

print(f"MSL Data Shape: {msl.shape}")
print(f"MSL Labels Shape: {msl_labels.shape}")

In [ ]:
print(f"MSL Total Telemetries: {len(msl_file_names)}")
print(f"MSL Features: {msl.shape[1]}")
print(f"MSL Total Anomaly Collective: {msl_total_anomaly_windows}")
print(f"MSL Total Anomaly Points: {np.sum(msl_labels == 1)}")
print(f"MSL Anomaly Rate: {(np.sum(msl_labels == 1) / msl_labels.shape[0]) * 100:.2f}%")

### **SMAP Dataset**

In the TAD-GAN research, the train and test datasets were concatenated.

In [ ]:
smap_split_path = os.path.join(nasa_dataset_dir, "smap.csv")
smap_split_url = f"{NASA_SPLIT_REPO}{NASA_SMAP_SPLIT}"
smap_split = pd.read_csv(smap_split_url)
smap_split.to_csv(smap_split_path)

# extract the file names corresponding to the SMAP dataset
smap_file_names = list(smap_split["chan_id"].values)
# add the .npy file extensions to the SMAP file names
smap_file_names = [f"{smap_file_name}.npy" for smap_file_name in smap_file_names]
print(smap_file_names)

In [ ]:
smap = []

for smap_file_name in smap_file_names:
    # load both the train and test time series for the same channel, concatenate
    # the time series and add them to the SMAP dataset
    smap_train_ts = np.load(os.path.join(nasa_train_path, smap_file_name))
    smap_test_ts = np.load(os.path.join(nasa_test_path, smap_file_name))
    smap_ts = np.concatenate((smap_train_ts, smap_test_ts), axis=0)
    
    smap.append(smap_ts)

print(f"SMAP Total Telemetry Channels: {len(smap)}")

In [ ]:
smap_labels = [None for _ in range(len(smap))]
smap_labels_data = nasa_labels[nasa_labels["spacecraft"] == "SMAP"]
smap_total_anomaly_windows = 0

for index, row in smap_labels_data.iterrows():
    # extract the telemetry from which the anomalies were obtained,
    # and search for its corresponding index in the SMAP dataset
    telemetry = f"{row['chan_id']}.npy"
    if telemetry not in smap_file_names:
        continue

    telemetry_idx = smap_file_names.index(telemetry) 
    if smap_labels[telemetry_idx] is None: 
        smap_labels[telemetry_idx] = np.zeros((smap[telemetry_idx].shape[0],))

    anomaly_sequences = json.loads(row["anomaly_sequences"])
    smap_total_anomaly_windows += len(anomaly_sequences)

    for start, end in anomaly_sequences:
        smap_labels[telemetry_idx][start:end + 1] = 1

smap_labels = np.hstack(smap_labels)
smap = np.concatenate(smap, axis=0)

print(f"SMAP Data Shape: {smap.shape}")
print(f"SMAP Labels Shape: {smap_labels.shape}")

In [ ]:
print(f"SMAP Total Telemetries: {len(smap_file_names)}")
print(f"SMAP Features: {smap.shape[1]}")
print(f"SMAP Total Anomaly Collective: {smap_total_anomaly_windows}")
print(f"SMAP Total Anomaly Points: {np.sum(smap_labels == 1)}")
print(f"SMAP Anomaly Rate: {(np.sum(smap_labels == 1) / smap_labels.shape[0]) * 100:.2f}%")

## **Yahoo S5**


This dataset is part of the [Yahoo Webscope Program](https://webscope.sandbox.yahoo.com/catalog.php?datatype=s&did=70) and contains four benchmarks:
- **A1 Benchmark**: with real data
- **A2 Benchmark**: with synthetic data
- **A3 Benchmark**: with synthetic data
- **A4 Benchmark**: with synthetic data

*A1-A2 Benchmarks*: contain real and synthetic time-series with labeled anomalies. The synthetic data set contains time-series with random seasonality, trend and noise. The outliers in the synthetic dataset are inserted at random positions. The fields:

- 0 timestamp
- 1 value
- 2 is_anomaly

*A3-A4 Benchmarks*: contain synthetic time-series. The A3Benchmark only contains outliers while the A4Benchmark also contains the anomalies that are marked as change-points. The synthetic time-series have varying noise and trends with three pre-specified seasonalities. The fields:

- 0 timestamps: the UNIX timestamp marks every hour
- 1 value: the value of time series at this timestamp
- 2 anomaly: 1 if this stamp is an outlier
- 3 changepoint: 1 if this stamp is a change point
- 4 trend: the additive trend value for this timestamp 
- 5 noise: the additive noise value for this timestamp
- 6 seasonality1: the 12-hour seasonality value
- 7 seasonality2: the daily seasonality value
- 8 seasonality3: the weekly seasonality value

In [ ]:
A1_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A1Benchmark/"
A1_FILE_NAME = "real_"
A1_N_FILES = 67

A2_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A2Benchmark/"
A2_FILE_NAME = "synthetic_"
A2_N_FILES = 100

A3_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A3Benchmark/"
A3_FILE_NAME = "A3Benchmark-TS"
A3_N_FILES = 100

A4_FOLDER = "https://raw.githubusercontent.com/harris0704/nbaData16-17/refs/heads/master/Yahoo_S5_Data/A4Benchmark/"
A4_FILE_NAME = "A4Benchmark-TS"
A4_N_FILES = 100

In [ ]:
yahoo_s5_datasets_dir = os.path.join(os.getcwd(), "datasets", "yahoo_s5")

if os.path.exists(yahoo_s5_datasets_dir):
    print("Folder for the Yahoo S5 datasets already exists!")
else:
    os.mkdir(yahoo_s5_datasets_dir)
    print("Folder for the Yahoo S5 datasets was succesfully created!")

### **A1 Dataset**

In [ ]:
yahoo_s1_a1_dataset_dir = os.path.join(yahoo_s5_datasets_dir, "A1")

if os.path.exists(yahoo_s1_a1_dataset_dir):
    print("Folder for the Yahoo S5 A1 Benchmark already exists!")
else:
    os.mkdir(yahoo_s1_a1_dataset_dir)
    print("Folder for the Yahoo S5 A1 Benchmark was succesfully created!")

In [ ]:
a1_points = 0
a1_anomalies = 0 

for i in range(1, A1_N_FILES + 1):
    # name of the .csv in the repository
    a1_ts_fn = f"{A1_FILE_NAME}{i}.csv"
    # url to the raw .csv file in the repository
    a1_ts_url = f"{A1_FOLDER}{a1_ts_fn}"
    # path to where the .csv file will be saved locally
    a1_ts_path = os.path.join(yahoo_s1_a1_dataset_dir, a1_ts_fn)

    a1_ts_df = pd.read_csv(a1_ts_url)
    a1_ts_df.to_csv(a1_ts_path)
    # count the number of points and the number of anomaly points
    a1_points += len(a1_ts_df)
    a1_anomalies += (a1_ts_df["is_anomaly"] == 1).sum()

    if i % 10 == 0:
        print(f"Downloaded {i} .csv files.")

print()
print("Finished downloading!")
print("---------------------")
print(f"A1 Total Signals: {A1_N_FILES}")
print(F"A1 Total Points: {a1_points}")
print(f"A1 Total Anomaly Points: {a1_anomalies}")
print(f"A1 Anomaly Rate: {(a1_anomalies / a1_points) * 100:.2f}%")

In [ ]:
(a1_ts_df["is_anomaly"] == 1).sum()

In [ ]:
a1_to_plot_ids = [20, 26, 48]
a1_to_plot_csvs = [f"{A1_FILE_NAME}{id}.csv" for id in a1_to_plot_ids]
a1_to_plots = [pd.read_csv(os.path.join(yahoo_s1_a1_dataset_dir, a1_to_plot_csv)) for a1_to_plot_csv in a1_to_plot_csvs]

for i, a1_to_plot in enumerate(a1_to_plots):
    plot_anomaly_df(a1_to_plot, save=True, save_jpg=f"a1_plot_{i}.svg")

### **A2 Dataset**

In [ ]:
yahoo_s1_a2_dataset_dir = os.path.join(yahoo_s5_datasets_dir, "A2")

if os.path.exists(yahoo_s1_a2_dataset_dir):
    print("Folder for the Yahoo S5 A2 Benchmark already exists!")
else:
    os.mkdir(yahoo_s1_a2_dataset_dir)
    print("Folder for the Yahoo S5 A2 Benchmark was succesfully created!")

In [ ]:
a2_points = 0
a2_anomalies = 0 

for i in range(1, A2_N_FILES + 1):
    # name of the .csv in the repository
    a2_ts_fn = f"{A2_FILE_NAME}{i}.csv"
    # url to the raw .csv file in the repository
    a2_ts_url = f"{A2_FOLDER}{a2_ts_fn}"
    # path to where the .csv file will be saved locally
    a2_ts_path = os.path.join(yahoo_s1_a2_dataset_dir, a2_ts_fn)

    a2_ts_df = pd.read_csv(a2_ts_url)
    a2_ts_df.to_csv(a2_ts_path)
    # count the number of points and the number of anomaly points
    a2_points += len(a2_ts_df)
    a2_anomalies += (a2_ts_df["is_anomaly"] == 1).sum()

    if i % 10 == 0:
        print(f"Downloaded {i} .csv files.")

print()
print("Finished downloading!")
print("---------------------")
print(f"A2 Total Signals: {A2_N_FILES}")
print(F"A2 Total Points: {a2_points}")
print(f"A2 Total Anomaly Points: {a2_anomalies}")
print(f"A2 Anomaly Rate: {(a2_anomalies / a2_points) * 100:.2f}%")

In [ ]:
a2_to_plot_ids = [f"{A2_FILE_NAME}{id}.csv" for id in np.random.randint(low=1, high=(A2_N_FILES + 1), size=3)]
a2_to_plots = [pd.read_csv(os.path.join(yahoo_s1_a2_dataset_dir, a2_to_plot_id)) for a2_to_plot_id in a2_to_plot_ids]
for i, a2_to_plot in enumerate(a2_to_plots):
    plot_anomaly_df(a2_to_plot, save=True, save_jpg=f"a2_plot_{i}.svg")

In [ ]:
a2_to_plots[0]

### **A3 Dataset**

In [ ]:
yahoo_s1_a3_dataset_dir = os.path.join(yahoo_s5_datasets_dir, "A3")

if os.path.exists(yahoo_s1_a3_dataset_dir):
    print("Folder for the Yahoo S5 A3 Benchmark already exists!")
else:
    os.mkdir(yahoo_s1_a3_dataset_dir)
    print("Folder for the Yahoo S5 A3 Benchmark was succesfully created!")

In [ ]:
a3_points = 0
a3_anomalies = 0 

for i in range(1, A3_N_FILES + 1):
    # name of the .csv in the repository
    a3_ts_fn = f"{A3_FILE_NAME}{i}.csv"
    # url to the raw .csv file in the repository
    a3_ts_url = f"{A3_FOLDER}{a3_ts_fn}"
    # path to where the .csv file will be saved locally
    a3_ts_path = os.path.join(yahoo_s1_a3_dataset_dir, a3_ts_fn)

    a3_ts_df = pd.read_csv(a3_ts_url)
    a3_ts_df = a3_ts_df.rename(columns={"anomaly": "is_anomaly", "timestamps": "timestamp"})
    a3_ts_df.to_csv(a3_ts_path)
    # count the number of points and the number of anomaly points
    a3_points += len(a3_ts_df)
    a3_anomalies += (a3_ts_df["is_anomaly"] == 1).sum()

    if i % 10 == 0:
        print(f"Downloaded {i} .csv files.")

print()
print("Finished downloading!")
print("---------------------")
print(f"A3 Total Signals: {A3_N_FILES}")
print(F"A3 Total Points: {a3_points}")
print(f"A3 Total Anomaly Points: {a3_anomalies}")
print(f"A3 Anomaly Rate: {(a3_anomalies / a3_points) * 100:.2f}%")

In [ ]:
a3_to_plot_ids = [f"{A3_FILE_NAME}{id}.csv" for id in np.random.randint(low=1, high=(A3_N_FILES + 1), size=3)]
a3_to_plots = [pd.read_csv(os.path.join(yahoo_s1_a3_dataset_dir, a3_to_plot_id)) for a3_to_plot_id in a3_to_plot_ids]
for a3_to_plot in a3_to_plots:
    plot_anomaly_df(a3_to_plot)

### **A4 Dataset**

In [ ]:
yahoo_s1_a4_dataset_dir = os.path.join(yahoo_s5_datasets_dir, "A4")

if os.path.exists(yahoo_s1_a4_dataset_dir):
    print("Folder for the Yahoo S5 A4 Benchmark already exists!")
else:
    os.mkdir(yahoo_s1_a4_dataset_dir)
    print("Folder for the Yahoo S5 A4 Benchmark was succesfully created!")

In [ ]:
a4_points = 0
a4_anomalies = 0 

for i in range(1, A4_N_FILES + 1):
    # name of the .csv in the repository
    a4_ts_fn = f"{A4_FILE_NAME}{i}.csv"
    # url to the raw .csv file in the repository
    a4_ts_url = f"{A4_FOLDER}{a4_ts_fn}"
    # path to where the .csv file will be saved locally
    a4_ts_path = os.path.join(yahoo_s1_a4_dataset_dir, a4_ts_fn)

    a4_ts_df = pd.read_csv(a4_ts_url)
    a4_ts_df = a4_ts_df.rename(columns={"anomaly": "is_anomaly", "timestamps": "timestamp"})
    a4_ts_df.to_csv(a4_ts_path)
    # count the number of points and the number of anomaly points
    a4_points += len(a4_ts_df)
    a4_anomalies += (a4_ts_df["is_anomaly"] == 1).sum()

    if i % 10 == 0:
        print(f"Downloaded {i} .csv files.")

print()
print("Finished downloading!")
print("---------------------")
print(f"A4 Total Signals: {A4_N_FILES}")
print(F"A4 Total Points: {a4_points}")
print(f"A4 Total Anomaly Points: {a4_anomalies}")
print(f"A4 Anomaly Rate: {(a4_anomalies / a4_points) * 100:.2f}%")

In [ ]:
a4_to_plot_ids = [f"{A4_FILE_NAME}{id}.csv" for id in np.random.randint(low=1, high=(A4_N_FILES + 1), size=3)]
a4_to_plots = [pd.read_csv(os.path.join(yahoo_s1_a4_dataset_dir, a4_to_plot_id)) for a4_to_plot_id in a4_to_plot_ids]
for a4_to_plot in a4_to_plots:
    plot_anomaly_df(a4_to_plot)

## **Numenta Anomaly Benchmark**

The data here includes multiple types of time series from various application domains, and in the study they picked five:
- *Art*
- *AdEx*
- *AWS*
- *Traf*
- *Tweets*

In [ ]:
NAB_REPO = "https://raw.githubusercontent.com/numenta/NAB/refs/heads/master/data/"

ART_FOLDER = "artificialWithAnomaly/"
ART_FILE_NAMES = [
    "art_daily_flatmiddle",
    "art_daily_jumpsdown",
    "art_daily_jumpsup",
    "art_daily_nojump",
    "art_increase_spike_density",
    "art_load_balancer_spikes"
]

AD_EX_FOLDER = "realAdExchange/"
AD_EX_FILE_NAMES = [
    "exchange-2_cpc_results",
    "exchange-2_cpm_results",
    "exchange-3_cpc_results",
    "exchange-3_cpm_results",
    "exchange-4_cpc_results",
    "exchange-4_cpm_results",
]

AWS_FOLDER = "realAWSCloudwatch/"
AWS_FILE_NAMES = [
    "ec2_cpu_utilization_24ae8d",
    "ec2_cpu_utilization_53ea38",
    "ec2_cpu_utilization_5f5533",
    "ec2_cpu_utilization_77c1ca",
    "ec2_cpu_utilization_825cc2",
    "ec2_cpu_utilization_ac20cd",
    "ec2_cpu_utilization_c6585a",
    "ec2_cpu_utilization_fe7f93",
    "ec2_disk_write_bytes_1ef3de",
    "ec2_disk_write_bytes_c0d644",
    "ec2_network_in_257a54",
    "ec2_network_in_5abac7",
    "elb_request_count_8c0756",
    "grok_asg_anomaly",
    "iio_us-east-1_i-a2eb1cd9_NetworkIn",
    "rds_cpu_utilization_cc0c53",
    "rds_cpu_utilization_e47b3b"
]

TRAFFIC_FOLDER = "realTraffic/"
TRAFFIC_FILE_NAMES = [
    "TravelTime_387",
    "TravelTime_451",
    "occupancy_6005",
    "occupancy_t4013",
    "speed_6005",
    "speed_7578",
    "speed_t4013"
]

TWEETS_FOLDER = "realTweets/"
TWEETS_FILE_NAMES = [
    "Twitter_volume_AAPL",
    "Twitter_volume_AMZN",
    "Twitter_volume_CRM",
    "Twitter_volume_CVS",
    "Twitter_volume_FB",
    "Twitter_volume_GOOG",
    "Twitter_volume_IBM",
    "Twitter_volume_KO",
    "Twitter_volume_PFE",
    "Twitter_volume_UPS"
]

LABELS = "https://raw.githubusercontent.com/numenta/NAB/refs/heads/master/labels/combined_windows.json"

In [ ]:
nab_datasets_dir = os.path.join(os.getcwd(), "datasets", "nab")

if os.path.exists(nab_datasets_dir) is False:
    os.mkdir(nab_datasets_dir)
    print("Folder for the NAB datasets was succesfully created!")
else:
    print("Folder for the NAB datasets already exists!")

In [ ]:
import urllib
labels_path = os.path.join(nab_datasets_dir, "labels.json")
urllib.request.urlretrieve(LABELS, labels_path)

with open(labels_path, "r") as lfp:
    labels = json.load(lfp)

In [ ]:
def download_datasets(
    folder: str,
    file_names: list[str],
    local_dir: str,
) -> None:
    global NAB_REPO

    nab_points = 0
    nab_anomalies = 0

    for i, fn in enumerate(file_names):
        # name of the .csv in the repository
        nab_ts_fn = f"{fn}.csv"
        # url to the raw .csv file in the repository
        nab_ts_url = f"{NAB_REPO}{folder}{nab_ts_fn}"
        # path to where the .csv file will be saved locally
        nab_ts_path = os.path.join(local_dir, nab_ts_fn)

        nab_ts_df = pd.read_csv(nab_ts_url)
        nab_ts_df["is_anomaly"] = 0

        # iterate through the contextualized anomalies for each .csv file,
        # and if the timestamp is inside one of the intervals, toggle the is_anomaly attribute
        for start, end in labels[f"{folder}{nab_ts_fn}"]:
            start = pd.Timestamp(start)
            end = pd.Timestamp(end)
            interval = pd.Interval(start, end, closed="both")

            nab_ts_df.loc[
                nab_ts_df["timestamp"].map(lambda ts: pd.Timestamp(ts) in interval),
                "is_anomaly"
            ] = 1

        nab_points += len(nab_ts_df)
        nab_anomalies += (nab_ts_df["is_anomaly"] == 1).sum()
        nab_ts_df.to_csv(nab_ts_path, index=False)

        if (i + 1) % 5 == 0:
            print(f"Downloaded {i + 1} .csv files.")

    print()
    print(f"Finished downloading {folder}!")
    print("---------------------")
    print(f"Total Signals: {len(file_names)}")
    print(F"Total Points: {nab_points}")
    print(f"Total Anomaly Points: {nab_anomalies}")
    print(f"Anomaly Rate: {(nab_anomalies / nab_points) * 100:.2f}%")

### **Artificial with Anomaly**

In [ ]:
nab_art_dataset_dir = os.path.join(nab_datasets_dir, "ART")

if os.path.exists(nab_art_dataset_dir):
    print("Folder for the NAB Artificial with Anomaly dataset already exists!")
else:
    os.mkdir(nab_art_dataset_dir)
    print("Folder for the NAB Artificial with Anomaly dataset was succesfully created!")

In [ ]:
download_datasets(folder=ART_FOLDER, file_names=ART_FILE_NAMES, local_dir=nab_art_dataset_dir)

### **Ad Exchange**

In [ ]:
nab_ad_ex_dataset_dir = os.path.join(nab_datasets_dir, "AD_EX")

if os.path.exists(nab_ad_ex_dataset_dir):
    print("Folder for the NAB Ad Ex already exists!")
else:
    os.mkdir(nab_ad_ex_dataset_dir)
    print("Folder for the NAB Ad Ex dataset was succesfully created!")

In [ ]:
download_datasets(folder=AD_EX_FOLDER, file_names=AD_EX_FILE_NAMES, local_dir=nab_ad_ex_dataset_dir)

In [ ]:
for i, ad_ex_file_name in enumerate(AD_EX_FILE_NAMES):
    ad_ex_df = pd.read_csv(os.path.join(nab_ad_ex_dataset_dir, f"{ad_ex_file_name}.csv"))
    plot_anomaly_df(ad_ex_df, save=True, save_jpg=f"ad_ex_plot_{i}.svg")

### **AWS**

In [ ]:
nab_aws_dataset_dir = os.path.join(nab_datasets_dir, "AWS")

if os.path.exists(nab_aws_dataset_dir):
    print("Folder for the NAB AWS dataset already exists!")
else:
    os.mkdir(nab_aws_dataset_dir)
    print("Folder for the NAB AWS dataset was succesfully created!")

In [ ]:
download_datasets(folder=AWS_FOLDER, file_names=AWS_FILE_NAMES, local_dir=nab_aws_dataset_dir)

### **Real Traffic**

In [ ]:
nab_traffic_dataset_dir = os.path.join(nab_datasets_dir, "TRAFFIC")

if os.path.exists(nab_traffic_dataset_dir):
    print("Folder for the NAB Traffic dataset already exists!")
else:
    os.mkdir(nab_traffic_dataset_dir)
    print("Folder for the NAB Traffic dataset was succesfully created!")

In [ ]:
download_datasets(folder=TRAFFIC_FOLDER, file_names=TRAFFIC_FILE_NAMES, local_dir=nab_traffic_dataset_dir)

In [ ]:
for i, traffic_file_name in enumerate(TRAFFIC_FILE_NAMES):
    traffic_df = pd.read_csv(os.path.join(nab_traffic_dataset_dir, f"{traffic_file_name}.csv"))
    traffic_df['value'] = 2 * (traffic_df['value'] - traffic_df['value'].min()) / (traffic_df['value'].max() - traffic_df['value'].min()) - 1
    plot_anomaly_df(traffic_df, save=True, save_jpg=f"traffic_plot_{i}.svg")

### **Tweets**

In [ ]:
nab_tweets_dataset_dir = os.path.join(nab_datasets_dir, "TWEETS")

if os.path.exists(nab_tweets_dataset_dir):
    print("Folder for the NAB Tweets dataset already exists!")
else:
    os.mkdir(nab_tweets_dataset_dir)
    print("Folder for the NAB Tweets dataset was succesfully created!")

In [ ]:
download_datasets(folder=TWEETS_FOLDER, file_names=TWEETS_FILE_NAMES, local_dir=nab_tweets_dataset_dir)